# Path B Pilot: STATE-ONLY 三分类（1 run 对比验证）

**目的**：跑 1 个对比 run，量化 multitask vs state-only 在同 setting 下的 `test_macro_f1_state` 差异，决定是否要做全套 state-only 重跑。

| Field | Value |
|-------|-------|
| Sweep | `configs/sweeps/pathB_cma_state_only_pilot.yaml` |
| Task | `cma_pathB_state_only`（单任务 state 三分类） |
| Head | `classification_seq` |
| Loss | `ce_seq_masked` |
| Early stopping | `val_macro_f1_state` |
| Setting | V+K+T / cross_subject / seed 0（**1 run**） |

**对比基准**：`runs/pathB_cma_state_trend/cross_subject/cma_pathB_state_trend_3seed/triple_video_km_event_telem_60hz/seed0` 的 `test_macro_f1_state` 字段。

**决策表（Cell 4 自动输出）**：

| Δ F1_state | 决策 |
|---|---|
| < 0.015 | 用现有 multitask 数据，0 重跑 |
| 0.015–0.04 | 重跑 path B + CMA + EFT-CDF state-only (~72 runs) |
| > 0.04 | 全套重跑 (~264 runs) |

In [ ]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyyaml', 'scipy', '-q'])

%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2: Dry run — verify plan = exactly 1 run, no import errors
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/pathB_cma_state_only_pilot.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathB_state_only_pilot \
    --dry_run 2>&1 | tail -10

In [ ]:
# Cell 3: Run the single pilot experiment (re-run safe — skips if already done)
# -u: unbuffered stdout so per-epoch progress prints in real time
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/pathB_cma_state_only_pilot.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathB_state_only_pilot

In [ ]:
# Cell 4: Compare state-only pilot vs existing multitask test_macro_f1_state
import json, glob
from pathlib import Path

# Pilot result (state-only)
PILOT_PATTERN = '/content/drive/MyDrive/AmuCS_experiment/runs/pathB_state_only_pilot/**/metrics.json'
pilot_files = glob.glob(PILOT_PATTERN, recursive=True)
assert pilot_files, f'No pilot metrics.json found at {PILOT_PATTERN}'
pilot = json.load(open(pilot_files[0]))

# Multitask baseline (same setting: V+K+T / cross / seed 0)
MT_PATTERN = ('/content/drive/MyDrive/AmuCS_experiment/runs/pathB_cma_state_trend/cross_subject/'
              'cma_pathB_state_trend_3seed/triple_video_km_event_telem_60hz/*seed0*/metrics.json')
mt_files = glob.glob(MT_PATTERN)
assert mt_files, f'No multitask seed0 found at {MT_PATTERN}'
mt = json.load(open(mt_files[0]))

# Pilot uses single-task naming (test_macro_f1_state) OR mean (test_macro_f1_mean) — both same in single-task
pilot_state_f1 = pilot.get('test_macro_f1_state', pilot.get('test_macro_f1_mean'))
pilot_state_bacc = pilot.get('test_balanced_acc_state', pilot.get('test_balanced_acc_mean'))
mt_state_f1 = mt['test_macro_f1_state']
mt_state_bacc = mt['test_balanced_acc_state']

delta_f1 = pilot_state_f1 - mt_state_f1
delta_bacc = pilot_state_bacc - mt_state_bacc

print('='*60)
print('STATE-ONLY (pilot)  vs  MULTITASK state column (existing)')
print('='*60)
print(f'  test_macro_f1_state    pilot={pilot_state_f1:.4f}  multitask={mt_state_f1:.4f}  Δ={delta_f1:+.4f}')
print(f'  test_balanced_acc_state pilot={pilot_state_bacc:.4f}  multitask={mt_state_bacc:.4f}  Δ={delta_bacc:+.4f}')
print()
print('Pilot tr/va/te F1_state:', pilot.get('train_macro_f1_state' if 'train_macro_f1_state' in pilot else 'train_macro_f1_mean'),
      pilot.get('val_macro_f1_state', pilot.get('val_macro_f1_mean')), pilot_state_f1)

print()
print('DECISION:')
abs_delta = abs(delta_f1)
if abs_delta < 0.015:
    print(f'  Δ={delta_f1:+.4f} < 0.015  →  USE EXISTING multitask data, 0 re-runs needed')
elif abs_delta < 0.04:
    print(f'  0.015 ≤ Δ={delta_f1:+.4f} < 0.04  →  RE-RUN path B + CMA + EFT-CDF state-only (~72 runs)')
else:
    print(f'  Δ={delta_f1:+.4f} ≥ 0.04  →  FULL re-run all baselines state-only (~264 runs)')